0.1

In [ ]:
import gc
import io
import json
import logging
import os
import time
import warnings
from pathlib import Path

import joblib
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap
import sklearn
import xgboost as xgb
from scipy.stats import spearmanr
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.ensemble import RandomForestClassifier
from sklearn.frozen import FrozenEstimator
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import GroupKFold, train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

0.2

In [ ]:
SEED = 42
np.random.seed(SEED)

# Skip raw dataset preprocessing
QUICK_RESUME = True

# Корень проекта. Path.cwd() — рабочая директория Jupyter.
# При необходимости переопредели вручную:
BASE_DIR = Path.cwd()
# BASE_DIR = Path('/path/to/project')

PATHS = {
    "legit": str(BASE_DIR / "legit_features.csv"),
    "dga_dir": str(BASE_DIR / "Results"),
    "dga_pattern": "*.csv",
    "legacy_full": str(BASE_DIR / "dataset_full.csv"),
    "raw": str(BASE_DIR / "dataset_raw.csv"),
    "raw_clean": str(BASE_DIR / "dataset_raw_clean.csv"),  # после очистки дубликатов
    "baseline_stats": str(BASE_DIR / "baseline_stats.json"),
    "processed": str(BASE_DIR / "dataset_processed.csv"),
    "splits": str(BASE_DIR / "split_indices.npz"),
    "scaler": str(BASE_DIR / "scaler.pkl"),
    "artifact": str(BASE_DIR / "model_artifact.json"),
    "plots_dir": str(BASE_DIR / "plots"),
    "models_dir": str(BASE_DIR / "models"),
}

# Целевые FPR для порога и отчёта (доли, не проценты)
TARGET_FPR_VAL = 1e-3
REPORT_FPRS = [1e-2, 1e-3, 1e-4]
TARGET_RECALLS = [0.95, 0.99]
REALISTIC_PREVALENCES = [1e-3, 1e-4]

# Доля семейств DGA, уходящих в out-of-distribution тест
HOLDOUT_FAMILY_FRAC = 0.20

# Подвыборки для дорогих операций (SHAP/permutation/latency)
SHAP_SAMPLE_N = 2000
PERMUTATION_SAMPLE_N = 20000
LATENCY_SAMPLES = 1000

# Минимальный размер семейства для holdout
MIN_FAMILY_SIZE_FOR_OOD = 5000

Path(PATHS["plots_dir"]).mkdir(exist_ok=True)
Path(PATHS["models_dir"]).mkdir(exist_ok=True)

assert Path(PATHS["legit"]).is_file(), f"Missing file: {PATHS['legit']}"
assert Path(PATHS["dga_dir"]).is_dir(), f"Not a directory: {PATHS['dga_dir']}"
print("BASE_DIR:", BASE_DIR)

BASE_DIR: d:\Data\Scripts


0.3

In [3]:
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")
warnings.filterwarnings("ignore", category=FutureWarning)
logging.getLogger("lightgbm").setLevel(logging.ERROR)
sns.set_style("whitegrid")

0.4

In [ ]:
if QUICK_RESUME:
    full_df = pd.read_csv(PATHS["processed"])
    feature_cols = [c for c in full_df.columns if c.startswith("nlp_")]
    isnan_cols = sorted([c for c in full_df.columns if c.startswith("isnan_")])
    feature_cols_final = feature_cols + isnan_cols
    X_all = full_df[feature_cols_final].astype("float32")  # memory fix
    y_all = full_df["label"]
    groups_all = full_df["class"].values
    print(f"✓ QUICK_RESUME: {PATHS['processed']}")
    print(
        f"  shape={full_df.shape} | features={len(feature_cols)}+{len(isnan_cols)}={len(feature_cols_final)}"
    )
    print(f"  labels={y_all.value_counts().to_dict()}")
    print("\n⚠ Cells 1.1–2.2 SKIPPED. Jump to cell 3.1.")
else:
    print("QUICK_RESUME=False — run cells 1.1 → 2.2 as usual")

1.1

In [ ]:
if Path(PATHS["legacy_full"]).exists():
    print(f"WARNING: {PATHS['legacy_full']} exists and is IGNORED.")
    print("         Legacy file already has fillna=0, it will NOT be used.")
    print(f"         Raw data with NaN will be written to: {PATHS['raw']}\n")

         Legacy file already has fillna=0, it will NOT be used.
         Raw data with NaN will be written to: d:\Data\Scripts\dataset_raw.csv



In [7]:
# Legit: allow 'class' but validate it
legit = pd.read_csv(PATHS["legit"])

if "class" in legit.columns:
    vals = legit["class"]
    non_null = vals.dropna()
    uniq = non_null.unique()
    print("legit_features.csv already has 'class' column:")
    print(
        f"  n_rows={len(legit)} | non-null={len(non_null)} | unique values={list(uniq)[:5]}"
    )
    bad = [v for v in uniq if v != "legit"]
    assert not bad, (
        f"legit_features.csv contains non-'legit' class values: {bad}. "
        "This file may be mixed with DGA data. Fix before continuing."
    )

# Normalize: overwrite / create with the literal 'legit'
legit["class"] = "legit"
print(f"legit: {len(legit)} rows, class='legit' enforced\n")

# DGA files
dga_files = sorted(Path(PATHS["dga_dir"]).glob(PATHS["dga_pattern"]))
assert dga_files, f"No DGA files found in {PATHS['dga_dir']}/{PATHS['dga_pattern']}"

print(f"DGA files: {len(dga_files)}")
for f in dga_files:
    print(f"  {f.name:45s} | {os.path.getsize(f) / 1e6:7.2f} MB")

# Schema consistency
# legit:                 domain + nlp_*            (class = 'legit' set above)
# features_<family>.csv: domain + nlp_* + class  (class = family label)
base_cols = set(legit.columns) - {"class"}
required = base_cols

schema_issues = []
family_summary = []
for f in dga_files:
    head = pd.read_csv(f)
    cols = set(head.columns)
    missing = required - cols
    extra = cols - required - {"class"}
    if missing or extra:
        schema_issues.append((f.name, "columns", sorted(missing), sorted(extra)))
        continue
    if "class" not in cols:
        schema_issues.append((f.name, "class", [], ["missing class column"]))
        continue
    uniq = head["class"].dropna().unique()
    if len(uniq) != 1:
        schema_issues.append(
            (
                f.name,
                "class",
                [],
                [f"expected 1 unique class, got {len(uniq)}: {list(uniq)[:5]}"],
            )
        )
        continue
    fam = uniq[0]
    if fam == "legit":
        schema_issues.append((f.name, "class", [], ["class=legit in a DGA file"]))
        continue
    family_summary.append((f.name, str(fam), len(head)))

if schema_issues:
    print("\nSchema issues:")
    for name, kind, missing, extra in schema_issues:
        print(f"  {name} [{kind}]: missing={missing}, extra={extra}")
    raise ValueError("Sources have incompatible schemas or invalid class columns.")

print(
    f"schema OK: {len(dga_files)} DGA files, all contain "
    f"{len(required)} base columns + exactly one class value"
)
for name, fam, n in family_summary[:5]:
    print(f"  {name:45s} class={fam!r:25s} n={n}")
if len(family_summary) > 5:
    print(f"  ... ({len(family_summary)} files total)")

# Concat
dga = pd.concat([pd.read_csv(f) for f in dga_files], ignore_index=True)
assert "class" in dga.columns
assert dga["class"].isna().sum() == 0, "NaN in class column after concat"

full_df = pd.concat([legit, dga], ignore_index=True)
full_df = full_df.sample(frac=1, random_state=SEED).reset_index(drop=True)

assert full_df["class"].isna().sum() == 0
assert (full_df["class"] == "legit").sum() == len(legit), (
    f"expected {len(legit)} legit rows, got {(full_df['class'] == 'legit').sum()}. "
    "Some DGA file likely contains class='legit'."
)

full_df.to_csv(PATHS["raw"], index=False)

print(f"\nfull_df shape: {full_df.shape}")
print(f"classes: {full_df['class'].nunique()} (incl. legit)")
print(f"saved -> {PATHS['raw']}")

legit_features.csv already has 'class' column:
  n_rows=1000000 | non-null=1000000 | unique values=['legit']
legit: 1000000 rows, class='legit' enforced

DGA files: 50
  features_alureon.csv                          |   41.52 MB
  features_banjori.csv                          |   62.37 MB
  features_bedep.csv                            |   41.99 MB
  features_ccleaner.csv                         |   21.36 MB
  features_chinad.csv                           |   40.97 MB
  features_corebot.csv                          |   42.66 MB
  features_cryptolocker.csv                     |   41.66 MB
  features_dircrypt.csv                         |   41.68 MB
  features_dyre.csv                             |   43.61 MB
  features_fobber-v1.csv                        |   41.90 MB
  features_fobber-v2.csv                        |   41.56 MB
  features_gozi-gpl.csv                         |   41.54 MB
  features_gozi-luther.csv                      |   41.71 MB
  features_gozi-nasa.csv               

1.2

In [8]:
feature_cols = [c for c in full_df.columns if c.startswith("nlp_")]
print("n_features:", len(feature_cols))

dup_domain = full_df["domain"].duplicated().sum()
print("duplicate domains:", dup_domain)

conflicts = full_df.groupby("domain")["class"].nunique()
print("domains with label conflicts:", (conflicts > 1).sum())

dup_rows = full_df.duplicated(subset=feature_cols).sum()
print("duplicate feature rows:", dup_rows)

print("\nclass distribution:")
print(full_df["class"].value_counts().to_string())

n_features: 130
duplicate domains: 18
domains with label conflicts: 18
duplicate feature rows: 62

class distribution:
class
legit           1000000
rovnix            40890
murofet-v2        40000
kraken-v1         30000
nymaim            30000
symmi             30000
matsnu            30000
banjori           30000
zeus-newgoz       30000
vawtrak-v1        25000
vawtrak-v2        25000
ramnit            20000
padcrypt          20000
gozi-luther       20000
fobber-v2         20000
qakbot            20000
dyre              20000
ranbyus-v2        20000
shiotob           20000
corebot           20000
dircrypt          20000
pushdo            20000
locky             20000
bedep             20000
pykspa-noise      20000
gozi-gpl          20000
simda             20000
alureon           20000
tinba             20000
gozi-rfc4343      20000
chinad            20000
cryptolocker      20000
necurs            20000
proslikefan       20000
qadars            20000
sisron            20000
fobber-v1  

1.3

In [12]:
nan_mask = full_df[feature_cols].isna()

total_nan = int(nan_mask.values.sum())
n_features_with_nan = int(nan_mask.any().sum())
nan_cols = [c for c in feature_cols if nan_mask[c].any()]  # ← добавлено

print(f"total NaN cells: {total_nan}")
print(f"features with NaN: {n_features_with_nan} / {len(feature_cols)}")

assert total_nan > 0, (
    "No NaN in full_df. NaN indicators (isnan_*) will NOT be created. "
    "Check that cell 1.1 actually read raw sources "
    "and not the legacy dataset_full.csv that already has fillna=0."
)

# Per-class NaN rate
nan_rate_legit = nan_mask[full_df["class"] == "legit"].mean()
nan_rate_dga = nan_mask[full_df["class"] != "legit"].mean()

nan_summary = pd.DataFrame(
    {
        "nan_legit": nan_rate_legit,
        "nan_dga": nan_rate_dga,
        "diff": (nan_rate_legit - nan_rate_dga).abs(),
    }
).sort_values("diff", ascending=False)

print("\nTop-20 features by |nan_legit - nan_dga|:")
print(nan_summary.head(20).round(4).to_string())

# Sanity: NaN rate should correlate negatively with domain length
domain_len = full_df["domain"].str.len()
nan_per_row = nan_mask.sum(axis=1)
corr = domain_len.corr(nan_per_row)
print(f"\ncorr(domain_len, n_nan_per_row) = {corr:.4f}  (expect negative)")
assert corr < 0, (
    "NaN should be more frequent in short domains. "
    "If not, NaN probably come from missing source columns, not from short lengths."
)

# Parity check: identical NaN patterns → redundant indicators
sig_to_cols = {}
for c in nan_cols:
    sig = nan_mask[c].values.tobytes()
    sig_to_cols.setdefault(sig, []).append(c)

print(f"\nunique NaN signatures: {len(sig_to_cols)} among {len(nan_cols)} features")
for group in sig_to_cols.values():
    if len(group) > 1:
        print(f"  identical NaN: {group}")

if "nlp_3g_tske" in nan_cols and "nlp_3g_tkur" in nan_cols:
    agree = (nan_mask["nlp_3g_tske"] == nan_mask["nlp_3g_tkur"]).mean()
    print(f"\ntske/tkur NaN agreement: {agree * 100:.4f}%")

total NaN cells: 55476
features with NaN: 4 / 130

Top-20 features by |nan_legit - nan_dga|:
               nan_legit  nan_dga    diff
nlp_3g_tske       0.0241   0.0036  0.0205
nlp_3g_tkur       0.0241   0.0036  0.0205
nlp_2g_tkur       0.0000   0.0000  0.0000
nlp_2g_tske       0.0000   0.0000  0.0000
nlp_1g_50p        0.0000   0.0000  0.0000
nlp_1g_25p        0.0000   0.0000  0.0000
nlp_1g_dst_ch     0.0000   0.0000  0.0000
nlp_1g_75p        0.0000   0.0000  0.0000
nlp_1g_dist       0.0000   0.0000  0.0000
nlp_1g_cov        0.0000   0.0000  0.0000
nlp_1g_dst_kl     0.0000   0.0000  0.0000
nlp_1g_dst_ma     0.0000   0.0000  0.0000
nlp_1g_e          0.0000   0.0000  0.0000
nlp_1g_ken        0.0000   0.0000  0.0000
nlp_1g_kur        0.0000   0.0000  0.0000
nlp_1g_dst_em     0.0000   0.0000  0.0000
nlp_1g_dst_eu     0.0000   0.0000  0.0000
nlp_1g_dst_ca     0.0000   0.0000  0.0000
nlp_1g_pro        0.0000   0.0000  0.0000
nlp_1g_pstd       0.0000   0.0000  0.0000

corr(domain_len, n_nan_p

1.3.1

In [ ]:
for pair in [("nlp_2g_tske", "nlp_2g_tkur"), ("nlp_3g_tske", "nlp_3g_tkur")]:
    a, b = pair
    if a in full_df.columns and b in full_df.columns:
        rho, _ = spearmanr(full_df[a], full_df[b], nan_policy="omit")
        print(f"{a} vs {b}: spearman rho = {rho:.4f}")

nlp_2g_tske vs nlp_2g_tkur: spearman rho = 0.9961
nlp_3g_tske vs nlp_3g_tkur: spearman rho = 0.9985


1.4

In [13]:
# Diagnose duplicate domains BEFORE fixing anything
domain_meta = full_df.groupby("domain").agg(
    n_rows=("class", "size"),
    classes=("class", lambda s: tuple(sorted(set(s)))),
)
dups = domain_meta[domain_meta["n_rows"] > 1].sort_values("n_rows", ascending=False)

print(f"domains with >1 row: {len(dups)}")
print(f"total rows involved: {int(dups['n_rows'].sum())}")


def categorize(classes):
    if "legit" in classes and len(classes) > 1:
        return "legit+DGA"
    if len(classes) > 1:
        return "DGA+DGA (multi-family)"
    return "DGA+DGA (same family)"


dups = dups.assign(kind=dups["classes"].apply(categorize))
print("\nconflict kinds:")
print(dups["kind"].value_counts().to_string())

print("\nfull list:")
print(dups[["n_rows", "classes", "kind"]].to_string())

domains with >1 row: 18
total rows involved: 36

conflict kinds:
kind
legit+DGA    18

full list:
                     n_rows                classes       kind
domain                                                       
aroundtravel.net          2          (legit, pizd)  legit+DGA
bicyclehistory.net        2          (legit, pizd)  legit+DGA
credit-repair.com         2        (legit, matsnu)  legit+DGA
darkhunt.net              2    (legit, suppobox-2)  legit+DGA
difange.com               2    (legit, vawtrak-v3)  legit+DGA
draftexpress.com          2        (legit, matsnu)  legit+DGA
elliman.com               2    (legit, vawtrak-v3)  legit+DGA
englishhistory.net        2          (legit, pizd)  legit+DGA
garden-floor.com          2        (legit, matsnu)  legit+DGA
heat-station.com          2        (legit, matsnu)  legit+DGA
horse-discount.com        2        (legit, matsnu)  legit+DGA
manufacturer.com          2        (legit, matsnu)  legit+DGA
missionworkshop.com       2     (g

1.4.1

In [18]:
for fam in ["gozi-nasa", "gozi-rfc4343"]:
    sub = full_df[full_df["class"] == fam]
    print(f"\n=== {fam} ({len(sub)} rows) ===")
    print(sub["domain"].sample(20, random_state=42).tolist())
    print(f"mean len: {sub['domain'].str.len().mean():.1f}")
    print(
        f"mean digit ratio: {sub['domain'].str.count(r'[0-9]').sum() / sub['domain'].str.len().sum():.3f}"
    )


=== gozi-nasa (19997 rows) ===
['plutoreviewinnov.com', 'openlaunchpla.com', 'fprivateearlysaidmore.com', 'unworkissueuranus.com', 'apridivisionthat.com', 'nasaannouncementnasamis.com', 'propulsionandthe.com', 'witdevelopedginstrum.com', 'butandearlierorderrese.com', 'lifeplutoseeapproach.com', 'surfacekuiper.com', 'organizationsmiss.com', 'daproposalsthe.com', 'cmoreduedistantavailable.com', 'missionlapronot.com', 'fromsystemsbhartmanwith.com', 'spacesengineers.com', 'andinnovdeputy.com', 'nasasuggestplatcostcould.com', 'earlierrestrthrough.com']
mean len: 23.5
mean digit ratio: 0.000

=== gozi-rfc4343 (19999 rows) ===
['optimizationuppecorres.com', 'extensionthatthisthetype.com', 'firstinlabelsstatus.com', 'intendedfordeter.com', 'systemttonlyrepresents.com', 'ainsensitivitycopyright.com', 'hoffmanrinputdnsnum.com', 'internetshowscmightany.com', 'forthtakingthedfinally.com', 'rrfcexamplespaces.com', 'avoidttreelabel.com', 'thdnsunliseenimplement.com', 'eastlakelabelclaugust.com', 'd

1.5

In [15]:
# Remove ambiguous domains
# Policy: if a domain has conflicting classes, drop ALL its rows.
# Rationale: no model can learn a label for such a domain, and if it lands
# in different splits it creates real leakage. 18 rows out of 2M is negligible.
n_before = len(full_df)
ambiguous_domains = set(dups.index)

if ambiguous_domains:
    dropped = full_df[full_df["domain"].isin(ambiguous_domains)]
    print(f"dropping {len(dropped)} rows across {len(ambiguous_domains)} domains")
    print(dropped[["domain", "class"]].to_string(index=False))
    full_df = full_df[~full_df["domain"].isin(ambiguous_domains)].reset_index(drop=True)

print(
    f"\nbefore: {n_before} | after: {len(full_df)} | removed: {n_before - len(full_df)}"
)

# --- Verify ---
assert full_df["domain"].duplicated().sum() == 0, "still have duplicate domains"
assert (full_df.groupby("domain")["class"].nunique() > 1).sum() == 0, (
    "still have conflicts"
)
print("✓ no duplicate domains, no label conflicts")

dropping 36 rows across 18 domains
             domain        class
        difange.com   vawtrak-v3
   heat-station.com       matsnu
   garden-floor.com       matsnu
        elliman.com        legit
 bicyclehistory.net         pizd
   manufacturer.com       matsnu
   heat-station.com        legit
   aroundtravel.net        legit
 horse-discount.com        legit
 sciencemission.com        legit
 englishhistory.net         pizd
 englishhistory.net        legit
   draftexpress.com        legit
missionworkshop.com    gozi-nasa
        elliman.com   vawtrak-v3
   draftexpress.com       matsnu
 horse-discount.com       matsnu
 theinformation.com        legit
       darkhunt.net        legit
   spacescience.com        legit
   satisfaction.com        legit
 readingrainbow.org       nymaim
 bicyclehistory.net        legit
   aroundtravel.net         pizd
 theinformation.com gozi-rfc4343
   manufacturer.com        legit
   spacescience.com    gozi-nasa
  credit-repair.com        legit
   garde

1.6

In [16]:
# Re-check feature duplicates AFTER domain fix
# Same feature vector with different classes = irreducible ambiguity (worse than domain dups).
# Same feature vector with same class = harmless noise, can be deduplicated.
dup_mask = full_df.duplicated(subset=feature_cols, keep=False)
dups_feat = full_df[dup_mask].copy()

print(f"rows with duplicate feature vectors: {len(dups_feat)}")
print(
    f"unique feature vectors repeating: {dups_feat.drop_duplicates(subset=feature_cols).shape[0]}"
)

conflict_sigs = pd.Index([])
if len(dups_feat):
    sig_to_classes = dups_feat.groupby(feature_cols)["class"].agg(
        lambda s: tuple(sorted(set(s)))
    )
    conflict_sigs = sig_to_classes[sig_to_classes.apply(len) > 1].index

    if len(conflict_sigs):
        print(
            f"\n⚠ {len(conflict_sigs)} feature vectors appear with DIFFERENT classes:"
        )
        for i, sig in enumerate(conflict_sigs[:10]):
            mask = pd.MultiIndex.from_frame(full_df[feature_cols]).isin([sig])
            samples = full_df.loc[mask, ["domain", "class"]].head(5)
            print(f"\nsig {i}:")
            print(samples.to_string(index=False))
    else:
        print("\n✓ all feature duplicates are within the same class — safe to dedupe")

rows with duplicate feature vectors: 88
unique feature vectors repeating: 44

✓ all feature duplicates are within the same class — safe to dedupe


1.7

In [19]:
# Resolve feature duplicates
n_before = len(full_df)

# 1. Drop rows whose feature vector appears with DIFFERENT classes (irreducible).
if len(conflict_sigs):
    mi = pd.MultiIndex.from_frame(full_df[feature_cols])
    drop_mask = mi.isin(conflict_sigs)
    print(f"dropping {drop_mask.sum()} rows with cross-class feature collisions")
    full_df = full_df[~drop_mask].reset_index(drop=True)
else:
    print("no cross-class feature collisions to drop")

# 2. Dedupe remaining exact duplicates (same class, identical features).
#    Keep first; this removes near-identical examples that add no information.
full_df = full_df.drop_duplicates(subset=feature_cols, keep="first").reset_index(
    drop=True
)

print(
    f"\nbefore: {n_before} | after: {len(full_df)} | removed: {n_before - len(full_df)}"
)

# Final integrity
assert full_df["domain"].duplicated().sum() == 0
assert full_df.duplicated(subset=feature_cols).sum() == 0
assert full_df["label"].isna().sum() == 0 if "label" in full_df.columns else True
print("✓ clean: no duplicate domains, no duplicate feature vectors")

no cross-class feature collisions to drop

before: 1999964 | after: 1999920 | removed: 44
✓ clean: no duplicate domains, no duplicate feature vectors


1.8

In [20]:
# Persist cleaned data + baseline diagnostics
# Пересчитываем nan_mask и dup-статистику на очищенном full_df
nan_mask_clean = full_df[feature_cols].isna()
fam_sizes = full_df.loc[full_df["class"] != "legit", "class"].value_counts()

baseline_stats = {
    "n_rows": len(full_df),
    "n_legit": int((full_df["class"] == "legit").sum()),
    "n_dga": int((full_df["class"] != "legit").sum()),
    "n_families": int(fam_sizes.shape[0]),
    "family_sizes": fam_sizes.to_dict(),
    "family_size_min": int(fam_sizes.min()),
    "family_size_max": int(fam_sizes.max()),
    "family_size_median": float(fam_sizes.median()),
    "n_nan_cells": int(nan_mask_clean.values.sum()),
    "n_features_with_nan": int(nan_mask_clean.any().sum()),
    "n_dup_domains_removed": len(ambiguous_domains),
    "n_feature_collisions_removed": int(
        len(conflict_sigs) if "conflict_sigs" in dir() else 0
    ),
    "removed_rows_total": int(2000000 - len(full_df)),
}
with open(PATHS["baseline_stats"], "w", encoding="utf-8") as f:
    json.dump(baseline_stats, f, ensure_ascii=False, indent=2, default=float)

# Сохраняем cleaned-версию отдельно, чтобы raw остался для воспроизведения
full_df.to_csv(PATHS["raw_clean"], index=False)

print("baseline_stats saved ->", PATHS["baseline_stats"])
print("cleaned data saved ->", PATHS["raw_clean"])
print(
    f"\nfinal: {len(full_df)} rows | {fam_sizes.shape[0]} DGA families | "
    f"{baseline_stats['n_features_with_nan']} features with NaN"
)

baseline_stats saved -> d:\Data\Scripts\baseline_stats.json
cleaned data saved -> d:\Data\Scripts\dataset_raw_clean.csv

final: 1999920 rows | 50 DGA families | 4 features with NaN


2.1

In [ ]:
# Пересчитываем nan_mask на всякий случай (не зависим от ячейки 1.3)
nan_mask = full_df[feature_cols].isna()
nan_cols = [c for c in feature_cols if nan_mask[c].any()]
assert len(nan_cols) > 0, "No NaN columns to build indicators for. See cell 1.3."

# Group features by identical NaN signature: only one indicator per group
sig_to_cols = {}
for c in nan_cols:
    sig = nan_mask[c].values.tobytes()
    sig_to_cols.setdefault(sig, []).append(c)

# One representative per unique NaN pattern
nan_cols_representatives = [cols[0] for cols in sig_to_cols.values()]
duplicates_skipped = [c for c in nan_cols if c not in nan_cols_representatives]

print(
    f"NaN features: {len(nan_cols)} | unique patterns: {len(nan_cols_representatives)}"
)
if duplicates_skipped:
    print(f"  skipping duplicate indicators: {duplicates_skipped}")
    for cols in sig_to_cols.values():
        if len(cols) > 1:
            print(f'    group kept="{cols[0]}"  mirrored_by={cols[1:]}')

isnan_to_add = [
    c for c in nan_cols_representatives if f"isnan_{c}" not in full_df.columns
]
print(f"new indicators to add: {len(isnan_to_add)}")
for c in isnan_to_add:
    full_df[f"isnan_{c}"] = nan_mask[c].astype(np.int8)

# Sentinel fill
# -1 for non-negative features (entropy, lengths, ratios, counts):
#   -1 is out of domain, distinguishable by trees.
#   0 for features with negative valid values (skewness, etc.).
sentinel_minus1, sentinel_zero = [], []
for c in nan_cols:
    if full_df[c].min(skipna=True) >= 0:
        full_df[c] = full_df[c].fillna(-1)
        sentinel_minus1.append(c)
    else:
        full_df[c] = full_df[c].fillna(0)
        sentinel_zero.append(c)

print(f"sentinel=-1 for {len(sentinel_minus1)} features")
print(f"sentinel= 0 for {len(sentinel_zero)} features")

assert full_df[feature_cols].isna().sum().sum() == 0, "NaN still present after fill"
print("NaN after fill:", int(full_df[feature_cols].isna().sum().sum()))

NaN features: 4 | unique patterns: 2
  skipping duplicate indicators: ['nlp_2g_tske', 'nlp_3g_tske']
    group kept="nlp_2g_tkur"  mirrored_by=['nlp_2g_tske']
    group kept="nlp_3g_tkur"  mirrored_by=['nlp_3g_tske']
new indicators to add: 2
sentinel=-1 for 4 features
sentinel= 0 for 0 features
NaN after fill: 0


2.2

In [ ]:
isnan_cols = sorted([c for c in full_df.columns if c.startswith("isnan_")])
feature_cols_final = feature_cols + isnan_cols

print(
    "n_base:",
    len(feature_cols),
    "| n_isnan:",
    len(isnan_cols),
    "| total:",
    len(feature_cols_final),
)

# Sanity: with NaN-signature deduplication in cell 2.1, the number of
# isnan_* columns equals the number of UNIQUE NaN patterns, not the
# number of NaN features. Recompute patterns here to validate.
nan_mask_check = full_df[feature_cols].isna()
assert nan_mask_check.values.sum() == 0, (
    "NaN still present in full_df[feature_cols] after cell 2.1 fill — "
    "something went wrong in 2.1. Re-run 2.1 from a fresh full_df."
)

unique_nan_patterns = len(
    {
        nan_mask_check[c].values.tobytes()
        for c in feature_cols
        if nan_mask_check[c].any()
    }
)
# after fill there is no NaN, so use the isnan_* markers to count patterns instead
isnan_patterns = set()
for c in isnan_cols:
    isnan_patterns.add(full_df[c].values.tobytes())
n_unique_isnan_patterns = len(isnan_patterns)

print(f"unique NaN patterns among isnan_* columns: {n_unique_isnan_patterns}")
assert n_unique_isnan_patterns == len(isnan_cols), (
    f"isnan_* columns are not independent: "
    f"{len(isnan_cols)} columns but only {n_unique_isnan_patterns} unique patterns. "
    "Cell 2.1 should have deduplicated by NaN signature."
)

# Check that each isnan_* column actually has both 0 and 1
for c in isnan_cols:
    n_ones = int(full_df[c].sum())
    assert 0 < n_ones < len(full_df), (
        f"{c} has {n_ones} ones — expected some 0s and some 1s"
    )
    print(f"  {c}: {n_ones} ones ({n_ones / len(full_df) * 100:.4f}%)")

# --- Labels and derived state ---
full_df["label"] = (full_df["class"] != "legit").astype(int)

X_all = full_df[feature_cols_final].astype("float32")
y_all = full_df["label"]
groups_all = full_df["class"].values

full_df.to_csv(PATHS["processed"], index=False)
print(f"\nsaved processed -> {PATHS['processed']}")
print(f"X_all: {X_all.shape} | y_all: {y_all.shape}")
print(f"label distribution: {y_all.value_counts().to_dict()}")

n_base: 130 | n_isnan: 2 | total: 132
unique NaN patterns among isnan_* columns: 2
  isnan_nlp_2g_tkur: 11 ones (0.0006%)
  isnan_nlp_3g_tkur: 27727 ones (1.3864%)

saved processed -> d:\Data\Scripts\dataset_processed.csv
X_all: (1999920, 132) | y_all: (1999920,)
label distribution: {1: 999982, 0: 999938}


3.1

In [ ]:
families = sorted(full_df.loc[full_df["label"] == 1, "class"].unique())
fam_counts = full_df.loc[full_df["label"] == 1, "class"].value_counts()

print("n_families:", len(families))
print(fam_counts.to_string())

3.2 - 3.3

In [ ]:
rng = np.random.RandomState(SEED)

# 1) Hold out families for test_ood
# Filter eligible families: too-small families give noisy OOD metrics.
fam_sizes_all = full_df.loc[full_df["label"] == 1, "class"].value_counts()
eligible = fam_sizes_all[fam_sizes_all >= MIN_FAMILY_SIZE_FOR_OOD].index.tolist()
ineligible = sorted(set(families) - set(eligible))

print(
    f"eligible for OOD (>= {MIN_FAMILY_SIZE_FOR_OOD} samples): "
    f"{len(eligible)} / {len(families)}"
)
if ineligible:
    print(f"ineligible: {ineligible}")
    print(f"  sizes: {fam_sizes_all[ineligible].to_dict()}")

assert len(eligible) >= 2, (
    f"Not enough large families ({len(eligible)}) for OOD. "
    f"Lower MIN_FAMILY_SIZE_FOR_OOD or check data."
)

n_holdout = max(2, round(HOLDOUT_FAMILY_FRAC * len(eligible)))
held_families = rng.choice(eligible, size=min(n_holdout, len(eligible)), replace=False)
print(f"\nheld-out families ({len(held_families)}): {sorted(held_families)}")
print("held-out sizes:")
print(fam_sizes_all[held_families].to_string())

# Sanity: test_ood shouldn't be dominated by one family
held_sizes = fam_sizes_all[held_families]
ratio = held_sizes.max() / held_sizes.min()
print(f"\nheld-out max/min size ratio: {ratio:.2f}")
if ratio > 3:
    print(
        "WARNING: test_ood dominated by one family — per-family metrics uneven; "
        "consider stratifying holdout by size bucket."
    )

held_mask = full_df["class"].isin(held_families).to_numpy()
dga_idx = full_df.index[full_df["label"] == 1].to_numpy()
legit_idx = full_df.index[full_df["label"] == 0].to_numpy()

dga_held = dga_idx[held_mask[dga_idx]]
dga_pool = dga_idx[~held_mask[dga_idx]]

# 2) DGA pool → train 70% / val 15% / test_id 15%
dga_train_idx, dga_rest = train_test_split(dga_pool, test_size=0.30, random_state=SEED)
dga_val_idx, dga_test_id = train_test_split(dga_rest, test_size=0.50, random_state=SEED)

# 3) Legit → train 70% / val 15% / test 15%
legit_train_idx, legit_temp = train_test_split(
    legit_idx, test_size=0.30, random_state=SEED
)
legit_val_idx, legit_test = train_test_split(
    legit_temp, test_size=0.50, random_state=SEED
)

# 4) Assemble. legit_test is SHARED between test_id and test_ood
train_idx = np.concatenate([legit_train_idx, dga_train_idx])
val_idx = np.concatenate([legit_val_idx, dga_val_idx])
test_id_idx = np.concatenate([legit_test, dga_test_id])
test_ood_idx = np.concatenate([legit_test, dga_held])

# 5) Sanity checks
assert set(legit_test).issubset(set(test_id_idx))
assert set(legit_test).issubset(set(test_ood_idx)), (
    "legit_test must be identical in test_id and test_ood "
    "so FPR and FP-per-million are directly comparable between the two tests"
)
legit_part_in_id = np.sort(test_id_idx[np.isin(test_id_idx, legit_test)])
assert np.array_equal(legit_part_in_id, np.sort(legit_test))

for a, b, msg in [
    (train_idx, test_id_idx, "train ∩ test_id"),
    (train_idx, test_ood_idx, "train ∩ test_ood"),
    (val_idx, test_id_idx, "val ∩ test_id"),
    (val_idx, test_ood_idx, "val ∩ test_ood"),
    (dga_held, dga_pool, "dga_held ∩ dga_pool (family leak)"),
]:
    n = len(np.intersect1d(a, b))
    assert n == 0, f"{msg} = {n}, expected 0"
print("split asserts OK")

# 6) Persist
np.savez(
    PATHS["splits"],
    train_idx=train_idx,
    val_idx=val_idx,
    test_id_idx=test_id_idx,
    test_ood_idx=test_ood_idx,
    held_families=np.array(held_families),
)
print(f"saved -> {PATHS['splits']}")

# 7) Summary
for name, idx in [
    ("train", train_idx),
    ("val", val_idx),
    ("test_id", test_id_idx),
    ("test_ood", test_ood_idx),
]:
    sub = full_df.loc[idx]
    n_legit = int((sub["label"] == 0).sum())
    n_dga = int((sub["label"] == 1).sum())
    n_fam = sub.loc[sub["label"] == 1, "class"].nunique()
    print(
        f"{name:10s} | n={len(idx):>8d} | legit={n_legit:>8d} "
        f"| dga={n_dga:>8d} | families_in_dga={n_fam}"
    )

3.4

In [ ]:
X_train, y_train = X_all.loc[train_idx], y_all.loc[train_idx]
X_val, y_val = X_all.loc[val_idx], y_all.loc[val_idx]
X_test_id, y_test_id = X_all.loc[test_id_idx], y_all.loc[test_id_idx]
X_test_ood, y_test_ood = X_all.loc[test_ood_idx], y_all.loc[test_ood_idx]

df_test_id = full_df.loc[test_id_idx].copy()
df_test_ood = full_df.loc[test_ood_idx].copy()

groups_train = groups_all[train_idx]
groups_val = groups_all[val_idx]

print("shapes:")
print(f"  X_train:    {X_train.shape}")
print(f"  X_val:      {X_val.shape}")
print(f"  X_test_id:  {X_test_id.shape}")
print(f"  X_test_ood: {X_test_ood.shape}")
print(f"groups_train unique: {len(np.unique(groups_train))}")
print(f"groups_val   unique: {len(np.unique(groups_val))}")

# Test set composition checks
ood_fam_counts = df_test_ood.loc[df_test_ood["label"] == 1, "class"].value_counts()
print("\ntest_ood DGA composition:")
print(ood_fam_counts.to_string())

id_fam_counts = df_test_id.loc[df_test_id["label"] == 1, "class"].value_counts()
print(
    f"\ntest_id: {len(id_fam_counts)} families present, "
    f"n_dga={int((df_test_id['label'] == 1).sum())}"
)
print(
    f"test_ood: {len(ood_fam_counts)} families present, "
    f"n_dga={int((df_test_ood['label'] == 1).sum())}"
)

assert (df_test_id["label"] == 0).sum() == (df_test_ood["label"] == 0).sum(), (
    "legit_test must be identical in test_id and test_ood"
)

del X_all
gc.collect()
print("X_all freed")

3.5

In [ ]:
# Optional: prune known-redundant features before modeling

# Diagnostic context (from cell 1.3 and 11.x in earlier runs):
#   - nlp_2g_tske / nlp_2g_tkur: spearman rho ≈ 0.996
#   - nlp_3g_tske / nlp_3g_tkur: spearman rho ≈ 0.998
#   - NaN patterns are identical (100% agreement)
#   These are KS vs Kuiper on the same n-gram distribution. Keeping both
#   dilutes SHAP/gain between functionally identical features and slows
#   inference without adding information.

# Toggling from True→False or False→True requires re-running:

APPLY_REDUNDANT_PRUNING = True
REDUNDANT_TO_DROP = ["nlp_2g_tske", "nlp_3g_tske"]  # drop KS, keep Kuiper

if APPLY_REDUNDANT_PRUNING:
    to_drop = [c for c in REDUNDANT_TO_DROP if c in X_train.columns]
    kept = [c for c in X_train.columns if c not in to_drop]
    print(f"before: {len(feature_cols_final)} features")
    print(f"dropping: {to_drop}")
    print(f"after:  {len(kept)} features")

    X_train = X_train[kept]
    X_val = X_val[kept]
    X_test_id = X_test_id[kept]
    X_test_ood = X_test_ood[kept]
    feature_cols_final = kept
else:
    print(f"DISABLED — keeping {len(feature_cols_final)} features")

4.1

In [ ]:
scaler = StandardScaler().fit(X_train)


def to_scaled(X_df):
    arr = scaler.transform(X_df).astype("float32")
    return pd.DataFrame(arr, columns=X_df.columns, index=X_df.index)


X_train_s = to_scaled(X_train)
X_val_s = to_scaled(X_val)
X_test_id_s = to_scaled(X_test_id)
X_test_ood_s = to_scaled(X_test_ood)

joblib.dump(scaler, PATHS["scaler"])
print("scaler fitted:", X_train_s.shape)

5.1

In [ ]:
class LogisticRegressionSGD(BaseEstimator, ClassifierMixin):
    """L2-регуляризованная LogReg, mini-batch SGD, sigmoid, clip + decay.
    predict_proba возвращает (n, 2) как у sklearn."""

    def __init__(
        self,
        lr=0.02,
        n_epochs=50,
        batch_size=1024,
        lambda_reg=5e-4,
        class_weight=None,
        decay=0.99,
        clip_grad=5.0,
        random_state=42,
        verbose=False,
    ):
        self.lr = lr
        self.n_epochs = n_epochs
        self.batch_size = batch_size
        self.lambda_reg = lambda_reg
        self.class_weight = class_weight
        self.decay = decay
        self.clip_grad = clip_grad
        self.random_state = random_state
        self.verbose = verbose

    @staticmethod
    def _sigmoid(z):
        z = np.clip(z, -500, 500)
        return 1.0 / (1.0 + np.exp(-z))

    @staticmethod
    def _add_bias(X):
        return np.c_[np.ones(X.shape[0]), X]

    def _loss(self, Xb, y, w):
        p = self._sigmoid(Xb @ self.weights)
        eps = 1e-15
        ll = -np.sum(w * (y * np.log(p + eps) + (1 - y) * np.log(1 - p + eps))) / len(y)
        return ll + 0.5 * self.lambda_reg * np.sum(self.weights[1:] ** 2)

    def fit(self, X, y):
        X = np.asarray(X, dtype=np.float32)
        y = np.asarray(y, dtype=np.float32)

        self.classes_ = np.unique(y)
        if len(self.classes_) != 2:
            raise ValueError(f"Expected 2 classes, got {self.classes_}")

        Xb = self._add_bias(X)
        n, f = Xb.shape
        self.weights = np.zeros(f)

        if self.class_weight:
            w = np.ones(n)
            for cls, cw in self.class_weight.items():
                w[y == cls] = cw
        else:
            w = np.ones(n)

        rng = np.random.RandomState(self.random_state)
        lr = self.lr
        self.loss_history_ = [self._loss(Xb, y, w)]

        for epoch in range(self.n_epochs):
            order = rng.permutation(n)
            Xs, ys, ws = Xb[order], y[order], w[order]
            for start in range(0, n, self.batch_size):
                end = min(start + self.batch_size, n)
                Xbt, ybt, wbt = Xs[start:end], ys[start:end], ws[start:end]
                p = self._sigmoid(Xbt @ self.weights)
                g = Xbt.T @ (wbt * (p - ybt)) / len(ybt)
                g[1:] += self.lambda_reg * self.weights[1:]
                if self.clip_grad is not None:
                    gn = np.linalg.norm(g)
                    if gn > self.clip_grad:
                        g *= self.clip_grad / gn
                self.weights -= lr * g
            lr *= self.decay
            self.loss_history_.append(self._loss(Xb, y, w))
            if self.verbose and epoch % 10 == 0:
                print(
                    f"epoch {epoch:3d} | loss {self.loss_history_[-1]:.6f} | lr {lr:.6f}"
                )
        return self

    def predict_proba(self, X):
        X = np.asarray(X, dtype=np.float32)
        p1 = self._sigmoid(self._add_bias(X) @ self.weights)
        return np.column_stack([1 - p1, p1])

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X)[:, 1] >= threshold).astype(int)

6.1

In [ ]:
def build_custom_logreg():
    return LogisticRegressionSGD(
        lr=0.02,
        n_epochs=50,
        batch_size=1024,
        lambda_reg=5e-4,
        decay=0.99,
        clip_grad=5.0,
        random_state=SEED,
        verbose=False,
    )


def build_sk_logreg():
    return LogisticRegression(
        C=0.5,
        max_iter=300,
        solver="saga",
        n_jobs=-1,
        random_state=SEED,
    )


def build_rf():
    return RandomForestClassifier(
        n_estimators=100,
        max_depth=15,
        min_samples_leaf=20,
        n_jobs=-1,
        random_state=SEED,
    )


def build_lgbm():
    return lgb.LGBMClassifier(
        n_estimators=2000,
        learning_rate=0.08,
        num_leaves=31,
        min_child_samples=200,
        feature_fraction=0.8,
        bagging_fraction=0.8,
        bagging_freq=5,
        reg_alpha=0.1,
        reg_lambda=1.0,
        importance_type="gain",
        random_state=SEED,
        verbose=-1,
        n_jobs=-1,
    )


def build_xgb():
    return xgb.XGBClassifier(
        n_estimators=2000,
        learning_rate=0.08,
        max_depth=6,
        min_child_weight=10,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        tree_method="hist",
        eval_metric="aucpr",
        early_stopping_rounds=200,
        random_state=SEED,
        n_jobs=-1,
    )


def build_mlp():
    return MLPClassifier(
        hidden_layer_sizes=(128, 64),
        activation="relu",
        alpha=1e-4,
        batch_size=2048,
        learning_rate_init=1e-3,
        max_iter=30,
        early_stopping=True,
        n_iter_no_change=5,
        validation_fraction=0.1,
        random_state=SEED,
    )


MODELS = {
    "CustomLogReg": {"builder": build_custom_logreg, "scale": True},
    "SkLogReg": {"builder": build_sk_logreg, "scale": True},
    "RandomForest": {"builder": build_rf, "scale": False},
    "LightGBM": {"builder": build_lgbm, "scale": False},
    "XGBoost": {"builder": build_xgb, "scale": False},
    "MLP": {"builder": build_mlp, "scale": True},
}

7.1

In [ ]:
def get_X(name, split):
    """Возвращает правильную версию X: scaled или raw, в зависимости от модели."""
    use_scale = MODELS[name]["scale"]
    return {
        "train": X_train_s if use_scale else X_train,
        "val": X_val_s if use_scale else X_val,
        "test_id": X_test_id_s if use_scale else X_test_id,
        "test_ood": X_test_ood_s if use_scale else X_test_ood,
    }[split]


trained_models = {}
train_time = {}
lgbm_evals = {}

for name, cfg in MODELS.items():
    print(f"\n=== {name} ===")
    model = cfg["builder"]()
    Xtr, Xva = get_X(name, "train"), get_X(name, "val")
    t0 = time.perf_counter()

    if name in ("CustomLogReg",):
        model.fit(Xtr.values, y_train.values)
    elif name == "LightGBM":
        model.fit(
            Xtr,
            y_train,
            eval_set=[(Xva, y_val)],
            eval_metric="average_precision",
            callbacks=[lgb.early_stopping(200), lgb.record_evaluation(lgbm_evals)],
        )
        print("best_iter:", model.best_iteration_)
    elif name == "XGBoost":
        model.fit(Xtr, y_train, eval_set=[(Xva, y_val)], verbose=False)
        print("best_iter:", model.best_iteration)
    else:
        model.fit(Xtr, y_train)

    train_time[name] = time.perf_counter() - t0
    trained_models[name] = model
    print(f"trained in {train_time[name]:.1f}s")

7.2

In [ ]:
if "valid_0" in lgbm_evals and "average_precision" in lgbm_evals["valid_0"]:
    ap = lgbm_evals["valid_0"]["average_precision"]
    best_i = int(np.argmax(ap))
    print(f"best AP={ap[best_i]:.5f} @ iter {best_i + 1}/{len(ap)}")
    plt.figure(figsize=(8, 4))
    plt.plot(ap)
    plt.axvline(best_i, color="r", ls="--")
    plt.xlabel("iteration")
    plt.ylabel("val AP")
    plt.title("LightGBM val AP")
    plt.tight_layout()
    plt.savefig(f"{PATHS['plots_dir']}/lgbm_val_ap.png", dpi=140)
    plt.show()

8.1

In [ ]:
def pick_threshold(y_val, y_prob, target_fpr=1e-3):
    """Максимизирует TPR при FPR <= target_fpr. Возвращает (thr, fpr@thr, tpr@thr)."""
    fpr, tpr, thr = roc_curve(y_val, y_prob)
    valid = fpr <= target_fpr
    if not valid.any():
        return 0.5, None, None
    idx = np.argmax(tpr * valid)
    return float(thr[idx]), float(fpr[idx]), float(tpr[idx])


def pick_threshold_at_recall(y_val, y_prob, target_recall=0.95):
    _, r, thr = precision_recall_curve(y_val, y_prob)
    idx = np.searchsorted(-r, -target_recall, side="right") - 1
    return float(thr[max(idx, 0)])

8.2

In [ ]:
thresholds = {}  # FPR-targeted
thresholds_recall = {}  # Recall-targeted
val_probs = {}

for name, model in trained_models.items():
    Xva = get_X(name, "val")
    y_prob = model.predict_proba(Xva)
    y_prob = y_prob[:, 1] if y_prob.ndim == 2 else y_prob
    val_probs[name] = y_prob

    thr, fpr_at, tpr_at = pick_threshold(y_val, y_prob, TARGET_FPR_VAL)
    thresholds[name] = thr

    thr_rec = pick_threshold_at_recall(y_val, y_prob, target_recall=0.95)
    thresholds_recall[name] = thr_rec

    print(
        f"{name:15s} | FPR-thr={thr:.4f} (TPR={tpr_at:.3f}) | Recall-thr={thr_rec:.4f}"
    )

9.1

In [35]:
def tpr_at_fpr(y_true, y_prob, target_fpr):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    idx = np.searchsorted(fpr, target_fpr, side="right") - 1
    return float(tpr[max(idx, 0)])


def precision_at_recall(y_true, y_prob, target_recall):
    p, r, _ = precision_recall_curve(y_true, y_prob)
    idx = np.searchsorted(-r, -target_recall, side="right") - 1
    return float(p[max(idx, 0)])


def fp_per_million(fpr):
    return fpr * 1_000_000


def precision_at_prevalence(tpr, fpr, prevalence):
    num = prevalence * tpr
    den = num + (1 - prevalence) * fpr
    return num / den if den > 0 else 0.0


def metrics_block(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_prob),
        "pr_auc": average_precision_score(y_true, y_prob),
        "tpr": tp / (tp + fn) if tp + fn else 0.0,
        "fpr": fp / (fp + tn) if fp + tn else 0.0,
        "fp_per_million": fp_per_million(fp / (fp + tn)) if fp + tn else 0.0,
        "brier": brier_score_loss(y_true, y_prob),
    }


def per_family_recall(df_test, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    out = {}
    for fam in df_test.loc[df_test["label"] == 1, "class"].unique():
        m = (df_test["class"] == fam).to_numpy()
        out[fam] = float((y_pred[m] == 1).mean())
    return out

9.2

In [ ]:
test_probs = {"id": {}, "ood": {}}

for name, model in trained_models.items():
    for split in ("id", "ood"):
        Xte = get_X(name, f"test_{split}")
        p = model.predict_proba(Xte)
        p = p[:, 1] if p.ndim == 2 else p
        test_probs[split][name] = p

print("probs computed:")
for split in ("id", "ood"):
    for name in test_probs[split]:
        print(f"  {split:3s} | {name:15s} | mean={test_probs[split][name].mean():.4f}")

9.3

In [ ]:
rows = []
for name in trained_models:
    m = metrics_block(y_test_id, test_probs["id"][name], thresholds[name])
    m["model"] = name
    m["tpr@fpr=1e-2"] = tpr_at_fpr(y_test_id, test_probs["id"][name], 1e-2)
    m["tpr@fpr=1e-3"] = tpr_at_fpr(y_test_id, test_probs["id"][name], 1e-3)
    m["tpr@fpr=1e-4"] = tpr_at_fpr(y_test_id, test_probs["id"][name], 1e-4)
    for r in TARGET_RECALLS:
        m[f"prec@rec={r}"] = precision_at_recall(y_test_id, test_probs["id"][name], r)
    for prev in REALISTIC_PREVALENCES:
        m[f"prec@prev={prev}"] = precision_at_prevalence(m["tpr"], m["fpr"], prev)
    rows.append(m)

metrics_id = pd.DataFrame(rows).set_index("model")
cols = [
    "accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc",
    "pr_auc",
    "tpr",
    "fpr",
    "fp_per_million",
    "brier",
    "tpr@fpr=1e-2",
    "tpr@fpr=1e-3",
    "tpr@fpr=1e-4",
    "prec@rec=0.95",
    "prec@rec=0.99",
    "prec@prev=0.001",
    "prec@prev=0.0001",
]
print("=== test_id ===")
print(metrics_id[cols].round(5).to_string())

9.4

In [ ]:
rows = []
for name in trained_models:
    m = metrics_block(y_test_ood, test_probs["ood"][name], thresholds[name])
    m["model"] = name
    m["tpr@fpr=1e-2"] = tpr_at_fpr(y_test_ood, test_probs["ood"][name], 1e-2)
    m["tpr@fpr=1e-3"] = tpr_at_fpr(y_test_ood, test_probs["ood"][name], 1e-3)
    m["tpr@fpr=1e-4"] = tpr_at_fpr(y_test_ood, test_probs["ood"][name], 1e-4)
    for r in TARGET_RECALLS:
        m[f"prec@rec={r}"] = precision_at_recall(y_test_ood, test_probs["ood"][name], r)
    for prev in REALISTIC_PREVALENCES:
        m[f"prec@prev={prev}"] = precision_at_prevalence(m["tpr"], m["fpr"], prev)
    rows.append(m)

metrics_ood = pd.DataFrame(rows).set_index("model")
print("=== test_ood (novel families) ===")
print(metrics_ood[cols].round(5).to_string())

9.5

In [ ]:
fam_recalls = {}
for name in trained_models:
    fam_recalls[name] = per_family_recall(
        df_test_ood, test_probs["ood"][name], thresholds[name]
    )

fam_df = pd.DataFrame(fam_recalls)
fam_df["mean"] = fam_df.mean(axis=1)
fam_df["type"] = fam_df.index.map(
    lambda f: (
        "wordlist"
        if f
        in [
            "matsnu",
            "pizd",
            "suppobox-1",
            "suppobox-2",
            "suppobox-3",
            "vawtrak-v3",
            "gozi-nasa",
            "gozi-rfc4343",
        ]
        else "other"
    )
)
fam_df = fam_df.sort_values("mean")
print("=== per-family recall on test_ood ===")
print(fam_df.round(4).to_string())

10.1

In [ ]:
calibrated = {}
for name, model in trained_models.items():
    Xva = get_X(name, "val")
    try:
        cal = CalibratedClassifierCV(
            FrozenEstimator(model),
            method="isotonic",
        )
        cal.fit(Xva, y_val)
        calibrated[name] = cal
        print(f"{name:15s} | calibrated")
    except Exception as e:  # noqa: BLE001
        print(f"{name:15s} | failed: {e}")
        calibrated[name] = model

10.2

In [ ]:
for split in ("id", "ood"):
    Xte_split = "test_" + split
    y_split = y_test_id if split == "id" else y_test_ood
    print(f"--- {split} ---")
    for name, model in trained_models.items():
        Xte = get_X(name, Xte_split)
        p_raw = test_probs[split][name]
        p_cal = calibrated[name].predict_proba(Xte)[:, 1]
        print(
            f"{name:15s} | Brier raw={brier_score_loss(y_split, p_raw):.5f} "
            f"| cal={brier_score_loss(y_split, p_cal):.5f}"
        )

10.3

In [ ]:
def reliability_plot(model_name, split="id", n_bins=15):
    y_true = y_test_id if split == "id" else y_test_ood
    p_raw = test_probs[split][model_name]
    Xte = get_X(model_name, "test_" + split)
    p_cal = calibrated[model_name].predict_proba(Xte)[:, 1]

    plt.figure(figsize=(6, 5))
    for label, p in [("raw", p_raw), ("calibrated", p_cal)]:
        frac_pos, mean_pred = calibration_curve(
            y_true, p, n_bins=n_bins, strategy="quantile"
        )
        plt.plot(mean_pred, frac_pos, marker="o", label=label)
    plt.plot([0, 1], [0, 1], "k--", alpha=0.4)
    plt.xlabel("mean predicted prob")
    plt.ylabel("fraction positives")
    plt.title(f"Reliability — {model_name} ({split})")
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"{PATHS['plots_dir']}/reliability_{model_name}_{split}.png", dpi=140)
    plt.show()


reliability_plot("LightGBM", "id")
reliability_plot("LightGBM", "ood")

11.1

In [ ]:
lgb_imp_gain = pd.Series(
    trained_models["LightGBM"].booster_.feature_importance(importance_type="gain"),
    index=feature_cols_final,
).sort_values(ascending=False)

print("=== Top-30 LightGBM gain ===")
print(lgb_imp_gain.head(30).round(2))

11.2

In [ ]:
rng = np.random.RandomState(SEED)
shap_idx = rng.choice(len(X_val), size=min(SHAP_SAMPLE_N, len(X_val)), replace=False)
X_shap = X_val.iloc[shap_idx]

explainer = shap.TreeExplainer(trained_models["LightGBM"])
shap_vals = explainer.shap_values(X_shap)
if isinstance(shap_vals, list):
    shap_vals = shap_vals[1]  # бинарная классификация
shap_importance = pd.Series(
    np.abs(shap_vals).mean(axis=0), index=feature_cols_final
).sort_values(ascending=False)

print("=== Top-30 SHAP ===")
print(shap_importance.head(30).round(4))

11.3

In [ ]:
perm_idx = rng.choice(
    len(X_val), size=min(PERMUTATION_SAMPLE_N, len(X_val)), replace=False
)
X_perm = X_val.iloc[perm_idx]
y_perm = y_val.iloc[perm_idx]

perm = permutation_importance(
    trained_models["LightGBM"],
    X_perm,
    y_perm,
    scoring="average_precision",
    n_repeats=5,
    random_state=SEED,
    n_jobs=-1,
)
perm_imp = pd.Series(perm.importances_mean, index=feature_cols_final).sort_values(
    ascending=False
)
print("=== Top-30 permutation ===")
print(perm_imp.head(30).round(5))

11.4

In [ ]:
top_k = 30
sets = {
    "gain": set(lgb_imp_gain.head(top_k).index),
    "shap": set(shap_importance.head(top_k).index),
    "perm": set(perm_imp.head(top_k).index),
}
print(f"intersection of all three top-{top_k}: {len(set.intersection(*sets.values()))}")
print(f"gain ∩ shap: {len(sets['gain'] & sets['shap'])}")
print(f"gain ∩ perm: {len(sets['gain'] & sets['perm'])}")
print(f"shap ∩ perm: {len(sets['shap'] & sets['perm'])}")

fig, axes = plt.subplots(1, 3, figsize=(16, 7))
for ax, (title, s) in zip(
    axes, [("Gain", lgb_imp_gain), ("SHAP", shap_importance), ("Perm", perm_imp)]
):
    top = s.head(20)[::-1]
    ax.barh(range(len(top)), top.values, color="steelblue")
    ax.set_yticks(range(len(top)))
    ax.set_yticklabels(top.index, fontsize=8)
    ax.set_title(f"Top-20 {title}")
    ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig(f"{PATHS['plots_dir']}/importance_consensus.png", dpi=140)
plt.show()

12.1

In [ ]:
def top_errors(model_name, split="ood", n=15):
    y_true = y_test_ood if split == "ood" else y_test_id
    df = df_test_ood if split == "ood" else df_test_id
    p = test_probs[split][model_name]
    pred = (p >= thresholds[model_name]).astype(int)

    fp_mask = (y_true.to_numpy() == 0) & (pred == 1)
    fn_mask = (y_true.to_numpy() == 1) & (pred == 0)

    fp = (
        df.loc[fp_mask, ["domain", "class"]]
        .assign(prob=p[fp_mask])
        .sort_values("prob", ascending=False)
    )
    fn = (
        df.loc[fn_mask, ["domain", "class"]]
        .assign(prob=p[fn_mask])
        .sort_values("prob", ascending=True)
    )

    print(f"\n=== {model_name} ({split}) ===")
    print(f"FP count = {len(fp)} | FN count = {len(fn)}")
    print("\nTop FP (most confident, worst):")
    print(fp.head(n).to_string(index=False))
    print("\nTop FN (most confident, worst):")
    print(fn.head(n).to_string(index=False))
    return fp, fn


fp_lgbm, fn_lgbm = top_errors("LightGBM", "ood", n=15)
fp_xgb, fn_xgb = top_errors("XGBoost", "ood", n=10)

12.2

In [ ]:
df_ood = df_test_ood.copy()
df_ood["pred_lgbm"] = (test_probs["ood"]["LightGBM"] >= thresholds["LightGBM"]).astype(
    int
)

fn_by_family = df_ood[(df_ood["label"] == 1) & (df_ood["pred_lgbm"] == 0)]
print("FN by family:")
print(fn_by_family["class"].value_counts())

# примеры FN доменов по каждому семейству
for fam in fn_by_family["class"].unique():
    samples = fn_by_family[fn_by_family["class"] == fam]["domain"].head(3).tolist()
    print(f"  {fam}: {samples}")

13.1

In [ ]:
def benchmark_single(model, X_one, n_warm=50, n=1000):
    for _ in range(n_warm):
        model.predict_proba(X_one)
    t0 = time.perf_counter()
    for _ in range(n):
        model.predict_proba(X_one)
    return (time.perf_counter() - t0) / n * 1000  # ms


def benchmark_batch(model, X_batch, n_warm=5, n=20):
    for _ in range(n_warm):
        model.predict_proba(X_batch)
    t0 = time.perf_counter()
    for _ in range(n):
        model.predict_proba(X_batch)
    return (time.perf_counter() - t0) / n  # s


latency = {}
for name, model in trained_models.items():
    Xte = get_X(name, "test_ood")
    X_one = Xte.iloc[:1]
    X_batch = Xte.iloc[:1024]

    ms_single = benchmark_single(model, X_one)
    s_batch = benchmark_batch(model, X_batch)

    _buf = io.BytesIO()
    joblib.dump(model, _buf)
    _size_mb = _buf.tell() / 1e6
    del _buf

    latency[name] = {
        "single_ms": ms_single,
        "batch_1024_s": s_batch,
        "throughput_domains_s": 1024 / s_batch,
        "model_size_mb": _size_mb,
    }
    print(
        f"{name:15s} | single={ms_single:7.3f} ms | batch1024={s_batch:6.3f} s | throughput={1024 / s_batch:8.0f} dom/s | size={latency[name]['model_size_mb']:.2f} MB"
    )

lat_df = pd.DataFrame(latency).T
print("\n=== latency summary ===")
print(lat_df.round(3).to_string())

14.1

In [51]:
def cv_groupkfold(build_fn, X_full, y_full, groups, n_splits=5, target_fpr=1e-3):
    """GroupKFold по группам (семействам DGA; legit — одна группа 'legit').
    Для каждой модели возвращает mean/std ROC-AUC, PR-AUC, TPR@FPR."""
    gkf = GroupKFold(n_splits=n_splits)
    rocs, prs, tprs = [], [], []

    for fold, (tr, te) in enumerate(gkf.split(X_full, y_full, groups)):
        model = build_fn()
        model.fit(X_full.iloc[tr], y_full.iloc[tr])
        p = model.predict_proba(X_full.iloc[te])[:, 1]
        y_true = y_full.iloc[te]
        rocs.append(roc_auc_score(y_true, p))
        prs.append(average_precision_score(y_true, p))
        tprs.append(tpr_at_fpr(y_true, p, target_fpr))

    return {
        "roc_auc_mean": np.mean(rocs),
        "roc_auc_std": np.std(rocs),
        "pr_auc_mean": np.mean(prs),
        "pr_auc_std": np.std(prs),
        f"tpr@fpr{target_fpr}_mean": np.mean(tprs),
        f"tpr@fpr{target_fpr}_std": np.std(tprs),
    }

14.2

In [ ]:
# CV setup
# CV runs on train + val + test_id (excludes OOD — test_ood is reserved
# for the final held-out family evaluation in cell 9.4).
cv_idx = np.concatenate([train_idx, val_idx, test_id_idx])
X_cv = full_df.loc[cv_idx, feature_cols_final].astype("float32")
y_cv = full_df.loc[cv_idx, "label"]
g_cv = full_df.loc[cv_idx, "class"].values.astype(object).copy()

# Fix: GroupKFold needs legit split into pseudo-groups
# 'legit' is a single ~1M-row group. When GroupKFold places it into one
# test fold, the remaining folds have zero legit samples → fit crashes with
# "only one class in the data". Solution: hash legit domains into N
# deterministic pseudo-groups so the splitter can distribute legit across
# folds. DGA families are untouched — each stays atomic in one fold.
N_LEGIT_GROUPS = 10
legit_mask = g_cv == "legit"
domain_cv = full_df.loc[cv_idx, "domain"].values
legit_hashes = pd.util.hash_array(domain_cv[legit_mask]) % N_LEGIT_GROUPS
g_cv[legit_mask] = np.array([f"legit_{h}" for h in legit_hashes])

n_dga_fam = len([g for g in np.unique(g_cv) if not str(g).startswith("legit_")])
print(
    f"GroupKFold groups: {len(np.unique(g_cv))} total "
    f"= {N_LEGIT_GROUPS} legit pseudo-groups + {n_dga_fam} DGA families"
)

# Sanity: every group has exactly one class
for g in np.unique(g_cv):
    classes = np.unique(y_cv[g_cv == g])
    assert len(classes) == 1, (
        f"Group {g} has mixed classes {classes} — "
        f"this should never happen (legit vs DGA only split by hash)."
    )
print("assert OK: each group is single-class")

# CV builders
# Models with scale=True in MODELS get Pipeline(scaler, clf).
# Inside each fold, the scaler fits only on the training portion —
# no leakage from the test fold. This matches the pipeline in cell 7.1
# and makes CV metrics comparable to cell 9.3.
#
# Trees (RF, LightGBM, XGBoost) are scale-invariant → no Pipeline needed.
cv_builders = {
    "CustomLogReg": lambda: Pipeline(
        [
            ("scaler", StandardScaler()),
            ("clf", build_custom_logreg()),
        ]
    ),
    "SkLogReg": lambda: Pipeline(
        [
            ("scaler", StandardScaler()),
            ("clf", build_sk_logreg()),
        ]
    ),
    "RandomForest": lambda: build_rf(),
    "LightGBM": lambda: build_lgbm(),
    "XGBoost": lambda: build_xgb(),
    "MLP": lambda: Pipeline(
        [
            ("scaler", StandardScaler()),
            ("clf", build_mlp()),
        ]
    ),
}

# Run CV
cv_results = {}
for name, builder in cv_builders.items():
    print(f"\nCV {name}...")
    t0 = time.perf_counter()
    cv_results[name] = cv_groupkfold(
        builder,
        X_cv,
        y_cv,
        g_cv,
        n_splits=5,
        target_fpr=TARGET_FPR_VAL,
    )
    print(f"  done in {time.perf_counter() - t0:.1f}s")

cv_df = pd.DataFrame(cv_results).T
print("\n=== GroupKFold CV (5 folds, family-aware) ===")
print(cv_df.round(5).to_string())

del X_cv, g_cv
gc.collect()
print("X_cv freed")

15.1

In [ ]:
for name, model in trained_models.items():
    joblib.dump(model, f"{PATHS['models_dir']}/{name}.pkl")
    joblib.dump(calibrated[name], f"{PATHS['models_dir']}/{name}_cal.pkl")

artifact = {
    "feature_cols": feature_cols_final,
    "base_feature_cols": feature_cols,
    "isnan_cols": isnan_cols,
    "thresholds": thresholds,
    "held_families": list(held_families),
    "target_fpr_val": TARGET_FPR_VAL,
    "seed": SEED,
    "metrics_id": metrics_id[cols].round(6).to_dict(orient="index"),
    "metrics_ood": metrics_ood[cols].round(6).to_dict(orient="index"),
    "latency": {k: {kk: float(vv) for kk, vv in v.items()} for k, v in latency.items()},
    "versions": {
        "sklearn": sklearn.__version__,
        "lightgbm": lgb.__version__,
        "xgboost": xgb.__version__,
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "shap": shap.__version__,
    },
}
with open(PATHS["artifact"], "w", encoding="utf-8") as f:
    json.dump(artifact, f, ensure_ascii=False, indent=2, default=float)

print("artifact saved:", PATHS["artifact"])
print("models saved to:", PATHS["models_dir"])

15.2

In [ ]:
def predict_domains(
    domains_df, model_name="LightGBM", calibrated=False, feature_pipeline=None
):
    """Применение обученной модели к DataFrame с теми же признаками.
    feature_pipeline — функция предобработки (индикаторы NaN + sentinel),
    если она не была применена заранее."""
    suffix = "_cal" if calibrated else ""
    model = joblib.load(f"{PATHS['models_dir']}/{model_name}{suffix}.pkl")
    scaler_loaded = (
        joblib.load(PATHS["scaler"]) if MODELS[model_name]["scale"] else None
    )

    X = domains_df[feature_cols_final]
    if feature_pipeline:
        X = feature_pipeline(X)
    if scaler_loaded is not None:
        X = pd.DataFrame(scaler_loaded.transform(X), columns=X.columns, index=X.index)

    prob = model.predict_proba(X)[:, 1]
    pred = (prob >= thresholds[model_name]).astype(int)
    return pd.DataFrame({"prob": prob, "pred": pred}, index=domains_df.index)

16.1

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, split in zip(axes[:, 0], ["id", "ood"]):
    y_true = y_test_id if split == "id" else y_test_ood
    for name in trained_models:
        fpr, tpr, _ = roc_curve(y_true, test_probs[split][name])
        auc = roc_auc_score(y_true, test_probs[split][name])
        ax.plot(fpr, tpr, label=f"{name} ({auc:.3f})")
    ax.plot([0, 1], [0, 1], "k--", alpha=0.4)
    ax.set_title(f"ROC — {split}")
    ax.set_xlabel("FPR")
    ax.set_ylabel("TPR")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

for ax, split in zip(axes[:, 1], ["id", "ood"]):
    y_true = y_test_id if split == "id" else y_test_ood
    for name in trained_models:
        p, r, _ = precision_recall_curve(y_true, test_probs[split][name])
        ap = average_precision_score(y_true, test_probs[split][name])
        ax.plot(r, p, label=f"{name} ({ap:.3f})")
    ax.set_title(f"PR — {split}")
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{PATHS['plots_dir']}/roc_pr_both_splits.png", dpi=140)
plt.show()

16.2

In [ ]:
for split in ("id", "ood"):
    y_true = y_test_id if split == "id" else y_test_ood
    n = len(trained_models)
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
    for ax, name in zip(axes, trained_models):
        pred = (test_probs[split][name] >= thresholds[name]).astype(int)
        cm = confusion_matrix(y_true, pred, labels=[0, 1])
        ConfusionMatrixDisplay(cm, display_labels=["Legit", "DGA"]).plot(
            ax=ax, cmap="Blues", colorbar=False
        )
        ax.set_title(f"{name}\n({split})", fontsize=10)
    plt.tight_layout()
    plt.savefig(f"{PATHS['plots_dir']}/cm_{split}.png", dpi=140)
    plt.show()

16.3

In [ ]:
summary = []
for name in trained_models:
    for split in ("id", "ood"):
        y_true = y_test_id if split == "id" else y_test_ood
        m = metrics_block(y_true, test_probs[split][name], thresholds[name])
        summary.append(
            {
                "model": name,
                "split": split,
                "f1": m["f1"],
                "pr_auc": m["pr_auc"],
                "tpr": m["tpr"],
                "fpr": m["fpr"],
                "fp_per_million": m["fp_per_million"],
                "tpr@fpr=1e-3": tpr_at_fpr(y_true, test_probs[split][name], 1e-3),
                "prec@prev=1e-3": precision_at_prevalence(m["tpr"], m["fpr"], 1e-3),
                "single_ms": latency[name]["single_ms"],
            }
        )

summary_df = pd.DataFrame(summary).set_index(["model", "split"])
print("=== Summary test_id vs test_ood ===")
print(summary_df.round(5).to_string())